In [1]:
#!/usr/bin/env python3
"""
MAPS CF CONCEPTS TO LINE ITEMS USING AI

Usage:
    uv run map_inc_concepts.py
"""
import os
from dotenv import load_dotenv
from datetime import datetime, date
from openai import AsyncOpenAI, OpenAI
import pandas as pd
import asyncio
import json
import time
import random
import tqdm

load_dotenv(override=True)
## MAP CONCEPTS TO LINE ITEMS


# CHANGE TO FINANCIAL STATEMENT YOU WANT TO MAP
from xbrl_mappings.income_statement_xbrl_mapping import INCOME_STATEMENT_MAPPING

# OLLAMA SETTINGS
# OLLAMA_BASE_URL = "http://172.17.112.1:11434/v1"
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# ollama_model = "deepseek-r1:8b"

openai = OpenAI()
openai_model = "gpt-5.2-2025-12-11"

google_api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_client = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = "gemini-3-flash-preview"

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

## CHOOSE OPENROUTER MODEL
#openrouter_model = "stepfun/step-3.5-flash:free"
#openrouter_model = "x-ai/grok-4.1-fast"
#openrouter_model = "openai/gpt-oss-120b"
#openrouter_model = "qwen/qwen3-32b:nitro"
openrouter_model = "arcee-ai/trinity-large-preview:free"
openrouter_decider_model = "anthropic/claude-sonnet-4.5"

#### USE EXTRA BODY TO FORCE OPENROUTER TO USE CEREBRAS FOR GPT-OSS-120B
# openrouter_extra_body={
#     "provider": {
#         "only": ["cerebras"],        # restrict to Cerebras
#         #"only": ["deepinfra"],        # restrict to Cerebras
#         "allow_fallbacks": False,    # fail instead of switching providers
#     },
# }  

In [2]:
"""Main execution function"""
#### THIS CODE IMPORTS MAPPINGS PREVIOUSLY EXPORTED FROM THE DATABASE
### AND CREATES A LIST OF DICTIONARIES
with open("income_concept_mappings.txt", "r") as f:
    income_concept_mappings = f.read()

import ast

# Parse the file content (it's a list of dictionaries)
income_concept_mappings_list = ast.literal_eval(income_concept_mappings)

# Extract ai_discovered_concepts from each dictionary in the list
ai_discovered_list = [
    {"concept": concept, "field_name": item["field_name"]}
    for item in income_concept_mappings_list
    if "ai_discovered_concepts" in item
    for concept in (
        item["ai_discovered_concepts"]
        if isinstance(item["ai_discovered_concepts"], list)
        else [item["ai_discovered_concepts"]]
    )
]

concepts_list = [d["concept"] for d in ai_discovered_list]

income_mapping_json = json.dumps(INCOME_STATEMENT_MAPPING)

In [3]:
print(concepts_list)

['us-gaap_AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount', 'crh_EarningsPerShareOtherDisclosureAbstract', 'us-gaap_NetIncomeLossPerOutstandingLimitedPartnershipUnitBasicNetOfTax', 'us-gaap_NetIncomeLossPerOutstandingLimitedPartnershipAndGeneralPartnershipUnitBasicAbstract', 'bk_EarningsPerShareBasicAndDilutedEPSAbstract', 'us-gaap_EarningsPerShareBasicAbstract', 'us-gaap_IncomeLossFromContinuingOperationsPerBasicShare', 'us-gaap_EarningsPerShareBasicOtherDisclosuresAbstract', 'us-gaap_ParticipatingSecuritiesDistributedAndUndistributedEarningsLossBasic', 'us-gaap_EarningsPerShareAbstract', 'ge_NetEarningsPerShareAbstract', 'cof_ParticipatingSecuritiesDistributedandUndistributedEarningsLossexcludingPreferredStockDividendsBasic', 'us-gaap_EarningsPerUnitAbstract', 'us-gaap_SupplementalIncomeStatementElementsAbstract', 'gehc_IncomeLossPerShareFromContinuingOperationsAbstract', 'has_EarningsPerShareBasicAndDilutedEPSAbstract', 'hrl_EarningsPerShareBasicAndDilutedEPSAb

In [ ]:
i = 0
grok_response = []
#for concept in concepts_list:


In [8]:
concept = concepts_list[1]

In [9]:

print(f"{concept}")

instructions = f""" You are a finance analyst and expert with SEC EDGAR filings and XBRL concepts.
You are given a EDGAR financial statement XBRL concept and 
a JSON file with predifined income statement line items. 
Your job is to map the given XBRL concept to the best match income statement line item in the 
predifined mapping JSON file. Return only the income statement line item that 
best matches the given XBRL concept. No extra verbage.
DO NOT RETURN ANYTHING OTHER THAN THE INCOME STATEMENT LINE ITEM. NO EXTRA VERBAGE. NO EXTRA TEXT. NO EXTRA COMMENTS.
NO BLOCK CODE. 

JSON file: {income_mapping_json}
XBRL concept: {concept}
"""

#i += 1 
rate_limit_delay: float = 1.0

gemini_response = gemini_client.chat.completions.create(
    model=gemini_model,
    messages=[{"role": "user", "content": instructions}],
    #NEED TO ADD EXTRA BODY TO FORCE OPENROUTER TO USE CEREBRAS FOR GPT-OSS-120B
    #extra_body=openrouter_extra_body
)

LLM1_response = gemini_response.choices[0].message.content

crh_EarningsPerShareOtherDisclosureAbstract


In [10]:
print(LLM1_response)

basic_eps
